# Developer Career Intelligence Platform - 04 Machine Learning & Career Matching

This notebook demonstrates:
1. Unsupervised Developer Archetype clustering with Scikit-learn KMeans & PCA.
2. Vector space cosine similarity career role recommendations.
3. Dynamic What-if Career Simulator with projected score deltas.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
from database.database import DatabaseManager
from analytics.developer_score import DeveloperIntelligenceScorer
from ml.archetype_clusterer import ArchetypeClusterer
from ml.career_recommender import CareerRecommender
from ml.skill_simulator import SkillPathSimulator

db = DatabaseManager('../data/developer_intelligence.db')
user = db.get_user_by_username('alex-datascientist')
repos = db.get_repositories(user['id'])
langs = db.get_languages(user['id'])
commits = db.get_commits(user['id'])
prs = db.execute_query('SELECT * FROM pull_requests WHERE user_id = ?', (user['id'],))
scores = DeveloperIntelligenceScorer.compute_scores(user, repos, langs, commits, prs)

# 1. Archetype Clustering & PCA
clusterer = ArchetypeClusterer()
archetype_res = clusterer.classify_developer(scores)
print(f"Predicted Archetype: {archetype_res['archetype']}")
print(f"2D PCA Projection: ({archetype_res['developer_coords_2d']['x']}, {archetype_res['developer_coords_2d']['y']})")

In [2]:
# 2. Vector Space Career Role Matching
career_recs = CareerRecommender.evaluate_all_roles(langs, repos)
pd.DataFrame(career_recs)[['rank', 'role_name', 'fit_percentage', 'strong_skills', 'skill_gaps']].head(5)

In [3]:
# 3. What-if Career Simulator: What if Alex learns Docker & Cloud?
sim_res = SkillPathSimulator.simulate_skill_acquisition(
    target_role='Machine Learning Engineer',
    acquired_skills=['Docker', 'Kubernetes', 'FastAPI'],
    languages_df=langs,
    repos_df=repos
)
print(f"Target Role: {sim_res['target_role']}")
print(f"Baseline Readiness: {sim_res['baseline_fit']}%")
print(f"Simulated Readiness: {sim_res['simulated_fit']}%")
print(f"Projected Delta: +{sim_res['delta_increase']}%")
print(sim_res['narrative'])